# Team 02 — Two-year-ahead forecast (Deliverable 5)

**What "two years ahead" means here:** we stand at a forecast date, know only the past, and predict **every month of
the next 24 months at once**. We do *not* predict one month, look at the answer, and move forward.

**Procedure:**
1. **Hold out the last 24 months.** Train only on months before them.
2. **Only use what would be known at the forecast date.** "Same month last year" is *not* known for months 13–24
   (that year has not happened yet), so this model drops it and uses **"same month two years ago"**, which is known
   for all 24 months.
3. Fit XGBoost with the loss that won in `model_comparison.ipynb`, plus a **quantile model** (10th / 50th / 90th
   percentile) for an 80% prediction interval.
4. Compare with the baseline **"same month two years ago"**, overall and for year 1 vs year 2.
5. Diagnostics: error over the horizon, residuals, segments, feature importance, interval coverage.
6. Refit on **all** data and produce the real forecast for the next 24 months after the data ends.

Charts are saved to `data/visuals/` (git-ignored — they show real numbers; never commit them).
**NDA:** summaries only; clear outputs before committing.

## 1. Setup and shared helpers

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from xgboost import XGBRegressor

pd.set_option('display.max_columns', 40)
plt.rcParams['figure.figsize'] = (10, 4)
TARGET = 'OD_PAX'
RANDOM_STATE = 42

repo_root = next((f for f in [Path.cwd(), *Path.cwd().parents]
                  if (f / 'DATA.md').is_file() and (f / 'data').is_dir()), None)
if repo_root is None:
    raise FileNotFoundError('Open this notebook from inside the project repository.')
feat_path = repo_root / 'data' / 'processed' / 'team02_features.csv'
if not feat_path.is_file():
    raise FileNotFoundError('Run notebooks/feature_set.ipynb first -- it creates data/processed/team02_features.csv')
df = pd.read_csv(feat_path, parse_dates=['period'])
for c in ['ORG_AIRPRT', 'DES_AIRPRT', 'ORG_CNTRY_CD', 'DES_CNTRY_CD', 'route']:
    df[c] = df[c].astype(str)

CAT = ['ORG_AIRPRT', 'DES_AIRPRT', 'ORG_CNTRY_CD', 'DES_CNTRY_CD']
LAGS = ['pax_same_month_last_year', 'pax_same_month_2_years_ago']
for c in LAGS:
    df['log_' + c] = np.log1p(df[c].clip(lower=0))      # log versions for the linear model

months = np.sort(df['period'].unique())
print(f'feature table loaded: {len(months)} months, {pd.Timestamp(months[0]):%Y-%m} -> {pd.Timestamp(months[-1]):%Y-%m}')

# ---- shared helpers --------------------------------------------------------------------
def metrics(y, p):
    y, p = np.asarray(y, float), np.asarray(p, float)
    return {'MAE': np.mean(np.abs(y - p)), 'RMSE': np.sqrt(np.mean((y - p) ** 2)),
            'WAPE': np.abs(y - p).sum() / np.abs(y).sum()}

def to_xgb(frame, cols, ref):
    """Categoricals as pandas categories with the TRAINING categories only (unseen -> missing)."""
    X = frame[cols].copy()
    for c in CAT:
        if c in cols:
            X[c] = pd.Categorical(X[c].where(X[c].isin(ref[c])), categories=ref[c])
    return X

def fit_xgb(train, cols, loss, val_months=6, **extra):
    """Fit XGBoost with early stopping on the last `val_months` of the training period (never the test months)."""
    ref = {c: sorted(train[c].unique()) for c in CAT}
    cut = np.sort(train['period'].unique())[-val_months]
    fit, val = train[train['period'] < cut], train[train['period'] >= cut]
    if loss == 'poisson':
        obj, yf, yv = 'count:poisson', fit[TARGET], val[TARGET]
    elif loss == 'log1p':
        obj, yf, yv = 'reg:squarederror', np.log1p(fit[TARGET].clip(lower=0)), np.log1p(val[TARGET].clip(lower=0))
    else:
        raise ValueError(loss)
    params = dict(objective=obj, n_estimators=2000, learning_rate=0.05, max_depth=8, min_child_weight=5,
                  subsample=0.8, colsample_bytree=0.8, tree_method='hist', enable_categorical=True,
                  max_cat_to_onehot=1, early_stopping_rounds=100, random_state=RANDOM_STATE)
    params.update(extra)
    m = XGBRegressor(**params)
    m.fit(to_xgb(fit, cols, ref), yf, eval_set=[(to_xgb(val, cols, ref), yv)], verbose=False)
    m.ref_, m.cols_, m.loss_ = ref, cols, loss
    return m

def predict_xgb(m, frame):
    p = m.predict(to_xgb(frame, m.cols_, m.ref_))
    return np.expm1(p).clip(min=0) if m.loss_ == 'log1p' else p.clip(min=0)

def lag_baseline(train, test, lag_col):
    """Seasonal naive: the lag value; if the route has none, the route's training median; else the overall median."""
    route_med = train.groupby('route')[TARGET].median()
    return (test[lag_col].fillna(test['route'].map(route_med)).fillna(train[TARGET].median())).to_numpy()

In [ ]:
LOSS = 'poisson'      # <-- set to the winner printed by model_comparison.ipynb ('poisson' or 'log1p')
HORIZON = 24
VIS = repo_root / 'data' / 'visuals'
VIS.mkdir(parents=True, exist_ok=True)

COLS_2Y = CAT + ['DEP_MONTH', 'month_sin', 'month_cos', 'months_since_start', 'is_international',
                 'pax_same_month_2_years_ago', 'route_seen_2_years_ago']     # no last-year lag on purpose
assert 'pax_same_month_last_year' not in COLS_2Y

origin = months[-HORIZON]
train, test = df[df['period'] < origin].copy(), df[df['period'] >= origin].copy()
test['horizon_month'] = ((test['period'].dt.year - pd.Timestamp(origin).year) * 12
                         + test['period'].dt.month - pd.Timestamp(origin).month + 1)
test['horizon_year'] = np.where(test['horizon_month'] <= 12, 'year 1', 'year 2')
train_months = len(months) - HORIZON
print(f'forecast date: {pd.Timestamp(origin):%Y-%m} | training months: {train_months} | test months: {HORIZON}')
if train_months < 36:
    print('WARNING: fewer than 3 years of training data -- the two-year claim is weak; say so in the report.')
print(f'training rows with a two-years-ago value: {train["route_seen_2_years_ago"].mean():.1%}')

## 2. Fit the point model and the interval model

In [ ]:
point = fit_xgb(train, COLS_2Y, LOSS)
test['pred'] = predict_xgb(point, test)

# Quantile model on log1p (quantiles survive the log transform), one model for the 10th / 50th / 90th percentile
q = fit_xgb(train, COLS_2Y, 'log1p', objective='reg:quantileerror', quantile_alpha=np.array([0.1, 0.5, 0.9]))
qp = np.expm1(q.predict(to_xgb(test, q.cols_, q.ref_))).clip(min=0)
qp.sort(axis=1)                                   # guard against crossing quantiles
test['p10'], test['p50'], test['p90'] = qp[:, 0], qp[:, 1], qp[:, 2]
test['baseline'] = lag_baseline(train, test, 'pax_same_month_2_years_ago')
print(f'point model: {point.best_iteration + 1} trees | quantile model: {q.best_iteration + 1} trees')

## 3. Testing results — model vs baseline, year 1 vs year 2, interval coverage

In [ ]:
rows = []
for part, g in [('all 24 months', test), *test.groupby('horizon_year')]:
    for name, col in [('XGBoost (' + LOSS + ')', 'pred'), ('Baseline: same month two years ago', 'baseline')]:
        rows.append({'period': part, 'model': name, **metrics(g[TARGET], g[col])})
res = pd.DataFrame(rows).set_index(['period', 'model'])
display(res.round({'MAE': 1, 'RMSE': 1, 'WAPE': 3}))

inside = (test[TARGET] >= test['p10']) & (test[TARGET] <= test['p90'])
print(f'80% interval: covers {inside.mean():.1%} of actual route-months (target is about 80%) | '
      f'median width relative to prediction: {((test["p90"] - test["p10"]) / test["p50"].replace(0, np.nan)).median():.2f}x')

total = test.groupby('period')[[TARGET, 'pred', 'baseline']].sum()
print(f'network-level total passengers over 24 months: model error {(total["pred"].sum() / total[TARGET].sum() - 1):+.1%}, '
      f'baseline error {(total["baseline"].sum() / total[TARGET].sum() - 1):+.1%}')

## 4. Visuals (saved to `data/visuals/`, never committed)

In [ ]:
hist = train.groupby('period')[TARGET].sum()
band = test.groupby('period')[['p10', 'p90']].sum()   # sum of route intervals: a rough, wide network band

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(hist.index, hist.values, color='grey', label='observed (training)')
ax.plot(total.index, total[TARGET], color='black', label='observed (held out)')
ax.plot(total.index, total['pred'], color='tab:blue', label='model forecast')
ax.plot(total.index, total['baseline'], color='tab:orange', ls='--', label='baseline (2 years ago)')
ax.fill_between(band.index, band['p10'], band['p90'], color='tab:blue', alpha=.15, label='80% band (approx.)')
ax.axvline(pd.Timestamp(origin), color='red', ls=':', label='forecast date')
ax.set_title('Total monthly passengers: observed vs two-year-ahead forecast'); ax.legend(fontsize=8)
fig.savefig(VIS / 'observed_vs_forecast.png', dpi=150, bbox_inches='tight'); plt.show()

by_h = test.groupby('horizon_month').apply(lambda g: pd.Series({
    'model': metrics(g[TARGET], g['pred'])['WAPE'], 'baseline': metrics(g[TARGET], g['baseline'])['WAPE']}))
ax = by_h.plot(marker='o', ms=3, title='Error by months ahead (WAPE)')
ax.set_xlabel('months after the forecast date'); ax.set_ylabel('WAPE')
plt.savefig(VIS / 'error_by_horizon.png', dpi=150, bbox_inches='tight'); plt.show()

resid = np.log1p(test[TARGET]) - np.log1p(test['pred'])
resid.clip(-3, 3).plot.hist(bins=60, title='Residuals on the log scale (actual vs predicted)')
plt.xlabel('log1p(actual) - log1p(predicted)  (> 0 means we under-forecast)')
plt.savefig(VIS / 'residuals.png', dpi=150, bbox_inches='tight'); plt.show()

In [ ]:
size = train.groupby('route')[TARGET].mean()
test['route_size'] = pd.qcut(test['route'].map(size), 3, labels=['small', 'medium', 'large']).astype(str)
test.loc[test['route'].map(size).isna(), 'route_size'] = 'new route'
test['segment_intl'] = test['is_international'].map({0.0: 'domestic', 1.0: 'international'}).fillna('unknown')
seg = []
for col in ['segment_intl', 'route_size']:
    for s, g in test.groupby(col):
        seg.append({'split': col, 'segment': s, 'share of passengers': g[TARGET].sum() / test[TARGET].sum(),
                    'model WAPE': metrics(g[TARGET], g['pred'])['WAPE'],
                    'baseline WAPE': metrics(g[TARGET], g['baseline'])['WAPE']})
seg = pd.DataFrame(seg).set_index(['split', 'segment'])
display(seg.round(3))
seg[['model WAPE', 'baseline WAPE']].plot.barh(title='Error by segment (WAPE)'); plt.tight_layout()
plt.savefig(VIS / 'error_by_segment.png', dpi=150, bbox_inches='tight'); plt.show()

imp = pd.Series(point.get_booster().get_score(importance_type='gain')).reindex(COLS_2Y).fillna(0)
(imp / imp.sum()).sort_values().plot.barh(title='What the model relies on (share of gain)'); plt.tight_layout()
plt.savefig(VIS / 'feature_importance.png', dpi=150, bbox_inches='tight'); plt.show()

## 5. The real forecast — next 24 months after the data ends

Refit on **all** months, then forecast the 24 months after the last month in the data, for every route that was
active in the last 12 months. The two-years-ago value for each future month is already in the data. Trees cannot
extrapolate a trend beyond what they saw, so long-run growth is a known limitation.

In [ ]:
final = fit_xgb(df, COLS_2Y, LOSS)
last = pd.Timestamp(months[-1])
active = df[df['period'] > last - pd.DateOffset(months=12)][['ORG_AIRPRT', 'DES_AIRPRT', 'route', 'ORG_CNTRY_CD',
                                                               'DES_CNTRY_CD', 'is_international']].drop_duplicates('route')
future_periods = pd.date_range(last + pd.DateOffset(months=1), periods=HORIZON, freq='MS')
fut = active.merge(pd.DataFrame({'period': future_periods}), how='cross')
start = df['period'].min()
fut['DEP_MONTH'] = fut['period'].dt.month
fut['month_sin'] = np.sin(2 * np.pi * fut['DEP_MONTH'] / 12)
fut['month_cos'] = np.cos(2 * np.pi * fut['DEP_MONTH'] / 12)
fut['months_since_start'] = (fut['period'].dt.year - start.year) * 12 + fut['period'].dt.month - start.month
past = df[['route', 'period', TARGET]].assign(period=lambda d: d['period'] + pd.DateOffset(years=2))
fut = fut.merge(past.rename(columns={TARGET: 'pax_same_month_2_years_ago'}), on=['route', 'period'], how='left')
fut['route_seen_2_years_ago'] = fut['pax_same_month_2_years_ago'].notna().astype(int)
fut['forecast'] = predict_xgb(final, fut)

ftot = fut.groupby('period')['forecast'].sum()
allhist = df.groupby('period')[TARGET].sum()
fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(allhist.index, allhist.values, color='black', label='observed')
ax.plot(ftot.index, ftot.values, color='tab:blue', label='forecast (next 24 months)')
ax.axvline(last, color='red', ls=':'); ax.legend(); ax.set_title('Two-year demand forecast — total monthly passengers')
fig.savefig(VIS / 'two_year_forecast.png', dpi=150, bbox_inches='tight'); plt.show()
last12 = allhist[allhist.index > last - pd.DateOffset(months=12)].sum()
print(f'forecast year 1 vs last observed 12 months: {ftot.iloc[:12].sum() / last12 - 1:+.1%} | '
      f'year 2 vs year 1: {ftot.iloc[12:].sum() / ftot.iloc[:12].sum() - 1:+.1%}')
fut.drop(columns=['month_sin', 'month_cos']).to_csv(repo_root / 'data' / 'processed' / 'team02_two_year_forecast.csv', index=False)
print('saved data/processed/team02_two_year_forecast.csv (git-ignored) and charts in data/visuals/')

## 6. Conclusions *(fill in after running — words and rounded numbers only)*

- **Does the model beat "same month two years ago"?** Overall, year 1, year 2:
- **How much worse is year 2 than year 1?**
- **Interval coverage (target about 80%):**
- **Segments with the biggest errors:**
- **Does the data support a two-year claim?** (how many years of history, routes without a two-years-ago value, trend):
- **Business use for American Airlines:** e.g. a route-level two-year demand baseline for capacity and fleet
  planning and for flagging routes where demand is growing or shrinking; not a replacement for short-term
  revenue-management forecasts.
- **Risks / what would be needed before use:** shocks the history does not contain, new routes with no history,
  no price or capacity data; retrain monthly and track error against this baseline.

### Before committing
Kernel → Restart & Clear Output, then save.